In [ ]:
import json
import pandas as pd
import urllib.request
import time
import datetime
import os


def fetch_binance_klines(symbol, interval, start_time, end_time=None, limit=1000):
    rows = []
    base_url = 'https://api.binance.com/api/v3/klines'
    while True:
        params = [
            f'symbol={symbol}',
            f'interval={interval}',
            f'startTime={start_time}',
            f'limit={limit}'
        ]
        if end_time is not None:
            params.append(f'endTime={end_time}')
        url = base_url + '?' + '&'.join(params)

        with urllib.request.urlopen(url) as response:
            data = json.loads(response.read().decode())

        if not data:
            break

        rows.extend(data)
        last_time = data[-1][0]

        if len(data) < limit:
            break

        start_time = last_time + 1
        time.sleep(0.2)

    return rows


def klines_to_df(klines):
    df = pd.DataFrame(
        klines,
        columns=[
            'OpenTime', 'Open', 'High', 'Low', 'Close', 'Volume',
            'CloseTime', 'QuoteAssetVolume', 'Trades',
            'TakerBuyBaseAssetVolume', 'TakerBuyQuoteAssetVolume', 'Ignore'
        ]
    )
    df['Timestamp'] = pd.to_datetime(df['CloseTime'], unit='ms')
    df = df[['Open', 'High', 'Low', 'Close', 'Timestamp']]
    df[['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].astype(float)
    return df


symbol = 'BTCUSDT'
interval = '5m'
start_time = int(datetime.datetime(2022, 1, 1, 0, 0).timestamp() * 1000)
end_time = int(time.time() * 1000)

klines = fetch_binance_klines(symbol, interval, start_time, end_time=end_time)

df = klines_to_df(klines)

os.makedirs('data', exist_ok=True)
df.to_csv('data/bitcoin2022to2026.csv', index=False)
df.head()

ModuleNotFoundError: No module named 'urllib2'

In [ ]:
import h5py
with h5py.File(''.join(['data/bitcoin2022to2026_wf.h5']), 'r') as hf:
    datas = hf['inputs'].value
    labels = hf['outputs'].value
    input_times = hf['input_times'].value
    output_times = hf['output_times'].value
    original_inputs = hf['original_inputs'].value
    original_outputs = hf['original_outputs'].value
    original_datas = hf['original_datas'].value

In [ ]:
original_inputs[0].shape

In [ ]:
df.to_csv('data/bitcoin2022to2026.csv',index=None)